# 4. Transformers

The Transformer architecture (Vaswani et al., 2017) replaced recurrence with **self-attention**. This notebook covers:
- **Scaled dot-product attention**
- **Multi-head attention**
- **Positional encoding**
- Building a simple Transformer encoder block from scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

## 4.1 Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Each query attends to all keys; scaling by $\sqrt{d_k}$ prevents softmax saturation.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

seq_len, d_k = 4, 8
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)
output, weights = scaled_dot_product_attention(Q, K, V)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights[0].detach().numpy(), cmap='Blues')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_title('Attention Weights')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4.2 Multi-Head Attention and Positional Encoding

**Multi-head**: project Q, K, V into $h$ heads, attend in parallel, concatenate.

**Positional encoding** (sinusoidal): $PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$, $PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        bs = Q.size(0)
        Q = self.W_q(Q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_k)
        return self.W_o(context)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Visualize positional encoding
pe = PositionalEncoding(d_model=64)
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pe.pe[0, :50, :].numpy().T, aspect='auto', cmap='RdBu')
ax.set_xlabel('Position')
ax.set_ylabel('Dimension')
ax.set_title('Positional Encoding (sinusoidal)')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4.3 Transformer Encoder Block

Each block: Multi-Head Self-Attention + Residual + LayerNorm, then FFN + Residual + LayerNorm.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.mha(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class SimpleTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128, n_layers=2, n_classes=2, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([TransformerEncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, n_classes)
    
    def forward(self, x):
        x = self.pos_enc(self.embed(x))
        for block in self.blocks:
            x = block(x)
        return self.fc(x.mean(dim=1))

model_tf = SimpleTransformerClassifier(vocab_size=100, n_classes=2)
dummy = torch.randint(0, 100, (4, 20))
print(f"Input: {dummy.shape} -> Output: {model_tf(dummy).shape}")
print(f"Parameters: {sum(p.numel() for p in model_tf.parameters()):,}")

## 4.4 Training on a Toy Task

Classify sequences of integers as "sorted" or "not sorted".

In [ ]:
def make_sort_data(n_samples=2000, seq_len=15, vocab_size=50):
    X, y = [], []
    for _ in range(n_samples):
        if np.random.rand() > 0.5:
            seq = sorted(np.random.randint(1, vocab_size, seq_len))
            y.append(1)
        else:
            seq = list(np.random.randint(1, vocab_size, seq_len))
            y.append(0)
        X.append(seq)
    return torch.tensor(X), torch.tensor(y)

X_train, y_train = make_sort_data(3000)
X_test, y_test = make_sort_data(500)

model_sort = SimpleTransformerClassifier(vocab_size=50, d_model=32, n_heads=4, d_ff=64, n_layers=2).to(device)
optimizer = torch.optim.Adam(model_sort.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses, accs = [], []
for epoch in range(40):
    model_sort.train()
    idx = torch.randint(0, len(X_train), (256,))
    optimizer.zero_grad()
    loss = criterion(model_sort(X_train[idx].to(device)), y_train[idx].to(device))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    model_sort.eval()
    with torch.no_grad():
        acc = (model_sort(X_test.to(device)).argmax(1).cpu() == y_test).float().mean().item()
    accs.append(acc)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>2}: Loss={loss.item():.4f}, Test Acc={acc:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.grid(True, alpha=0.3)
ax2.plot(accs, color='green')
ax2.set_title('Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- **Self-attention** allows each token to attend to all others in $O(n^2)$ time, but is fully parallelizable
- **Multi-head attention** lets the model learn different types of relationships simultaneously
- **Positional encoding** is necessary since attention is permutation-invariant
- **Residual connections** and **layer normalization** are crucial for training deep transformers
- Transformers are the backbone of GPT, BERT, Vision Transformers (ViT), and most modern architectures